In [18]:
!pip install sentence-transformers mistralai

In [19]:
from textwrap import dedent
import kagglehub
import pandas as pd
from google.colab import drive
from mistralai import Mistral
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm

tqdm.pandas()

# Load dataset

In [20]:
def load_tweets_data() -> pd.DataFrame:
  """
  Load the Tweets dataset from either Google Drive or Kaggle.

  This function first attempts to load the dataset from a predefined
  Google Drive path. If that fails (e.g., the file or drive is not available),
  it falls back to downloading the dataset from Kaggle using the
  `kagglehub` package.

  Returns:
      pd.DataFrame: A DataFrame containing the tweet data from `Tweets.csv`.

  Raises:
      Exception: If both the Google Drive path and Kaggle download fail.
  """
  file_name = "Tweets.csv"
  try:
    file_path_drive = "_Inge/Demo/data"
    drive.mount("/content/drive")
    file_path = f"/content/drive/MyDrive/{file_path_drive}/{file_name}"
    return pd.read_csv(file_path)
  except:
    kaggle_dataset_name = "crowdflower/twitter-airline-sentiment"
    dataset_path = kagglehub.dataset_download(kaggle_dataset_name)
    return pd.read_csv(f"{dataset_path}/{file_name}")

In [21]:
data = load_tweets_data()
print("Number of messages: ", len(data))
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'twitter-airline-sentiment' dataset.
Number of messages:  14640


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing about it,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [22]:
data = data.sample(frac=0.1, random_state=31).reset_index(drop=True)
print("Number of messages:", len(data))

Number of messages: 1464


In [23]:
pd.set_option("display.max_colwidth", None)
data = data[["text", "airline_sentiment"]]
print(len(data))
data.head(10)

1464


,text,airline_sentiment
0,@united employees almost seem happy when delivery terrible customer service.,negative
1,@united Pleased to be a Premier Platinum,positive
2,"@united It's the horrible attitude from staff after , not just the delay. Not the level of service or respect one expects from United",negative
3,"@USAirways I did, but it won't help.. Can't believe you wouldn't take full fare for first class and gave away to Platinum Member.. Profits?",negative
4,@JetBlue ugh always know a way to my heart 😘🙌,positive
5,@AmericanAir cut it. Put me on a flt tomorrow.,negative
6,@AmericanAir Flight 4606 from MEM to DCA delayed 6 hours! Now holding breath while they keep me trapped 3 hours to de-ice. #scareair,negative
7,@USAirways nightmare trying to get to Costa Rica from PHL today. Stuck in Miami and no one answers at 800 number.,negative
8,@SouthwestAir Karen with customer service was very helpful. Thank you for providing one bright spot in a frustrating situation.,positive
9,"@united still missing my luggage, was promised someone would call, no call so far, flight from Shanghai to DC with connecting flight in ORD",negative


# APPROACH 1: EMBEDDINGS

## Transform texts into vectors

In [24]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(data["text"].tolist(), show_progress_bar=True, normalize_embeddings=True,)
embedding_df = pd.DataFrame(embeddings, columns=[f"embedding_{i}" for i in range(embeddings.shape[1])])
data = pd.concat([data, embedding_df], axis=1)
data.head()

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

,text,airline_sentiment,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,@united employees almost seem happy when delivery terrible customer service.,negative,-0.011297,0.048211,0.047031,-0.031304,-0.025107,-0.073602,0.014503,-0.084504,...,0.014358,-0.069147,-0.009844,-0.083051,0.022483,0.064934,0.023584,0.031794,0.019333,0.041993
1,@united Pleased to be a Premier Platinum,positive,0.001914,-0.000548,-0.058602,-0.047558,0.032342,0.056725,0.096571,0.017618,...,0.048327,0.045090,-0.033969,-0.018533,-0.079919,0.042597,0.075028,-0.074578,0.021241,0.047520
2,"@united It's the horrible attitude from staff after , not just the delay. Not the level of service or respect one expects from United",negative,0.025466,-0.018990,0.010425,-0.064791,0.052324,0.014387,0.012494,-0.092092,...,0.055487,0.012753,0.017789,-0.090627,0.018910,0.049752,-0.018638,-0.055850,-0.066755,0.061421
3,"@USAirways I did, but it won't help.. Can't believe you wouldn't take full fare for first class and gave away to Platinum Member.. Profits?",negative,-0.034568,-0.009168,0.006600,-0.018155,0.011799,0.021394,0.041008,0.045642,...,-0.022711,-0.006190,-0.021553,0.008327,-0.108208,-0.058103,0.010438,-0.131119,-0.105700,0.039594
4,@JetBlue ugh always know a way to my heart 😘🙌,positive,-0.001216,-0.008216,0.020240,0.059351,0.059404,0.021923,0.096180,-0.044549,...,-0.015286,0.094522,0.034099,-0.031135,0.020977,0.010896,0.028057,-0.020283,-0.037021,0.013070


## Train/test split

In [25]:
data_train, data_test = train_test_split(data, test_size=0.2,
                                         stratify=data["airline_sentiment"],
                                         random_state=31)

X_train = data_train.drop(["text", "airline_sentiment"], axis=1)
y_train = data_train["airline_sentiment"]

X_test = data_test.drop(["text", "airline_sentiment"], axis=1)
y_test = data_test["airline_sentiment"]

print("Size of train:", len(y_train))
print("Size of test:", len(y_test))

Size of train: 1171
Size of test: 293


## Train model

In [26]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## Evaluate model

In [27]:
y_pred = classifier.predict(X_test)
data_test = data_test[["text", "airline_sentiment"]].reset_index(drop=True)
data_test["prediction_embeddings"] = y_pred
data_test.head(10)

,text,airline_sentiment,prediction_embeddings
0,@SouthwestAir lost my sunglasses &amp; case on a flight 3933 last night - is there a lost &amp; found?,negative,negative
1,@VirginAmerica add DTW and I'm sold!,neutral,neutral
2,@united Got it. I am following the United page.,neutral,neutral
3,@united yes there is when you keep getting the same robotic answer.,negative,negative
4,@JetBlue I do follow you!,neutral,positive
5,“@JetBlue: Our fleet's on fleek. http://t.co/tEjrN09tfw” NOOOOOOOOOOOOOOOOOO,neutral,neutral
6,@USAirways then drop your fee! $150 ($300RT!!!!) makes no sense; most EU airlines charge $0. @AirCanada charges a reasonable $50.,negative,negative
7,@united i have a weekend of dealing with your company that would say otherwise.,negative,negative
8,@USAirways nightmare trying to get to Costa Rica from PHL today. Stuck in Miami and no one answers at 800 number.,negative,negative
9,"@USAirways I don't want a reservation change, just adding a service. Am going public with this to get help.",negative,negative


In [28]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7645

Classification Report:
              precision    recall  f1-score   support

    negative       0.81      0.95      0.87       184
     neutral       0.62      0.44      0.52        63
    positive       0.69      0.48      0.56        46

    accuracy                           0.76       293
   macro avg       0.71      0.62      0.65       293
weighted avg       0.75      0.76      0.75       293



# APPROACH 2: LARGE LANGUAGE MODEL (LLM)

In [29]:
model = "mistral-small-2503"
api_key = "TWIGPW4wfA7LEmB6OQLPgFDAIf7jsjO9"

client = Mistral(api_key=api_key)

def call_model(query: str) -> str:
  """
  Send a query to the chat model and return its response.

  This function uses the global `client` object to call the chat
  completion API with the specified model. The query is sent as a
  user message, and the first response message is returned.

  Args:
      query (str): The user input or prompt to send to the model.

  Returns:
      str: The content of the model's first response message.

  Raises:
      Exception: If the client call fails or no response is returned.
  """
  chat_response = client.chat.complete(
  model = model,
  messages = [{"role": "user", "content": query}])
  return chat_response.choices[0].message.content

In [30]:
call_model("is the following message positive, negative or neutral?: I will use this airline again because the coffee is perfect. thank you!")

'The message is **positive**. The use of phrases like "I will use this airline again" and "the coffee is perfect" along with "thank you!" indicates satisfaction and a positive experience.'

## Generate prompt for text classification

In [31]:
prompt = "I will provide a tweet that someone wrote to an airline. Analyse the sentiment of the tweet and return only one word: positive, negative or neutral. Provided tweet: "

In [32]:
call_model(prompt + "I will always use this airline again because it is the best one")

'positive'

In [33]:
data_test.head()

,text,airline_sentiment,prediction_embeddings
0,@SouthwestAir lost my sunglasses &amp; case on a flight 3933 last night - is there a lost &amp; found?,negative,negative
1,@VirginAmerica add DTW and I'm sold!,neutral,neutral
2,@united Got it. I am following the United page.,neutral,neutral
3,@united yes there is when you keep getting the same robotic answer.,negative,negative
4,@JetBlue I do follow you!,neutral,positive


In [34]:
data_test["prediction_llm"] = data_test["text"].progress_apply(lambda x: call_model(prompt + x))
data_test.head()

100%|██████████| 293/293 [01:24<00:00,  3.46it/s]


,text,airline_sentiment,prediction_embeddings,prediction_llm
0,@SouthwestAir lost my sunglasses &amp; case on a flight 3933 last night - is there a lost &amp; found?,negative,negative,negative
1,@VirginAmerica add DTW and I'm sold!,neutral,neutral,positive
2,@united Got it. I am following the United page.,neutral,neutral,neutral
3,@united yes there is when you keep getting the same robotic answer.,negative,negative,negative
4,@JetBlue I do follow you!,neutral,positive,neutral


In [35]:
y_pred = data_test["prediction_llm"]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7509

Classification Report:
              precision    recall  f1-score   support

    negative       0.81      0.97      0.88       184
     neutral       0.49      0.43      0.46        63
    positive       0.88      0.33      0.48        46

    accuracy                           0.75       293
   macro avg       0.73      0.57      0.60       293
weighted avg       0.75      0.75      0.73       293



## Improve prompt

In [36]:

prompt_improved = dedent("""
    You are an expert sentiment analyst specializing in customer service interactions with airlines.
    Your task is to classify the sentiment of a tweet directed at an airline.

    Follow these steps (think through them privately before answering, but only output the final classification word):
    1. Read the tweet carefully.
    2. Consider whether the tone is positive, negative, or neutral.
    3. Output exactly one word: positive, negative, or neutral. Do not add anything else.

    Examples:
    Tweet: "Thank you @Delta for the smooth flight, best crew ever!"
    Answer: positive

    Tweet: "My flight with @United was delayed for 6 hours and no updates were given."
    Answer: negative

    Tweet: "@AmericanAir what time does flight 123 depart from JFK?"
    Answer: neutral

    Now classify the following tweet:
""")

In [37]:
data_test["prediction_llm2"] = data_test["text"].progress_apply(lambda x: call_model(prompt_improved + x))
data_test.head()

100%|██████████| 293/293 [01:38<00:00,  2.99it/s]


,text,airline_sentiment,prediction_embeddings,prediction_llm,prediction_llm2
0,@SouthwestAir lost my sunglasses &amp; case on a flight 3933 last night - is there a lost &amp; found?,negative,negative,negative,negative
1,@VirginAmerica add DTW and I'm sold!,neutral,neutral,positive,neutral
2,@united Got it. I am following the United page.,neutral,neutral,neutral,neutral
3,@united yes there is when you keep getting the same robotic answer.,negative,negative,negative,negative
4,@JetBlue I do follow you!,neutral,positive,neutral,neutral


In [38]:
y_pred = data_test["prediction_llm2"]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7782

Classification Report:
              precision    recall  f1-score   support

    negative       0.84      0.96      0.90       184
     neutral       0.54      0.59      0.56        63
    positive       0.94      0.33      0.48        46

    accuracy                           0.78       293
   macro avg       0.77      0.62      0.65       293
weighted avg       0.79      0.78      0.76       293

